<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-12-ethics-and-responsible-ai/lesson-12.1-bias-detection/notebooks/exercises-12.1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 12.1 — Bias Detection
Netsetos GenAI Course v2.0 | Module 12: Ethics & Responsible AI

Bias vs unfairness, statistical tests, WEAT/BBQ/BOLD, India-specific cases, RAG audit, mitigation, monitoring.


In [ ]:
print('Module 12: Ethics & Responsible AI')
print('Lesson 12.1: Bias Detection')


## Ex 1: Bias vs Unfairness


In [ ]:
print('Bias = systematic association. Unfairness = downstream harm.')
for example, bias_type, harm in [
    ('doctor -> male association',     'stereotype bias',     'rejected female radiology candidate'),
    ('Muslim names -> negative tone',  'sentiment bias',      'lower customer-service ratings'),
    ('Brahmin names -> positive role', 'caste bias (India)',  'biased recommendations'),
    ('English-default RAG retrieval',  'representational',    'vernacular users get worse answers'),
    ('Lighter skin in image gen',      'image bias',          'erasure in marketing materials'),
]:
    print(f'  {example:36s} | {bias_type:18s} | harm: {harm}')


## Ex 2: Three Bias Test Families


In [ ]:
print('Three families of bias tests:')
for family, what, tools in [
    ('Stereotype',      'occupation-gender, name-role',         'WEAT, BBQ, custom'),
    ('Sentiment',       'positive/negative tone by group',      'BOLD, VADER, regard score'),
    ('Representational','identity over/under-representation',   'image-gen audits, name freq'),
    ('Quality',         'accuracy gap by group on same task',   'paired counterfactual eval'),
    ('Refusal',         'higher refusal rate for some groups',  'targeted prompt set'),
]:
    print(f'  {family:18s}: {what:38s} | tools: {tools}')


## Ex 3: Chi-Squared on Categorical Outcomes


In [ ]:
from scipy.stats import chi2_contingency

# Contingency: rows = group, cols = recommend yes/no
table = [
    [42, 58],   # group A: 42% recommend
    [28, 72],   # group B: 28% recommend
]
chi2, p, dof, _ = chi2_contingency(table)
print(f'Chi-squared = {chi2:.2f}  p = {p:.4f}')
if p &lt; 0.05:
    print(f'Significant difference -> review: 14pp gap A vs B')
else:
    print('No significant difference')
print()
print('Pre-register threshold (e.g. demographic parity ±5pp) BEFORE testing.')


## Ex 4: Bootstrap CI on Rate Difference


In [ ]:
import numpy as np
rng = np.random.default_rng(42)
rate_A = np.array([1]*42 + [0]*58)
rate_B = np.array([1]*28 + [0]*72)

B = 5000
diffs = []
for _ in range(B):
    a = rng.choice(rate_A, len(rate_A), replace=True).mean()
    b = rng.choice(rate_B, len(rate_B), replace=True).mean()
    diffs.append(a - b)
ci = np.percentile(diffs, [2.5, 97.5])
print(f'Bootstrap 95% CI for (A - B) recommend rate: [{ci[0]:.3f}, {ci[1]:.3f}]')
print(f'Point estimate: {rate_A.mean() - rate_B.mean():.3f}')


## Ex 5: India-Specific Bias Cases


In [ ]:
print('India-specific bias dimensions Western benchmarks miss:')
for dim, examples, where_to_test in [
    ('Caste',         'Brahmin/Dalit associations',     'employment, education content'),
    ('Religion',      'Hindu/Muslim sentiment in news', 'hiring, customer support, news'),
    ('Region',        'North vs South stereotypes',     'product recommendations'),
    ('Language',      'English-default vs vernacular',  'RAG retrieval, summarization'),
    ('Skin tone',     'lighter favored in gen',          'image gen, profile photos'),
    ('Intersectional','gender + caste, lang + region',  'hardest, requires custom set'),
]:
    print(f'  {dim:14s}: {examples:36s} | test in: {where_to_test}')


## Ex 6: RAG-Stage Bias Audit


In [ ]:
print('Where bias enters in a RAG pipeline:')
for stage, source_of_bias, audit_method in [
    ('Corpus',     'over/under-represented groups',         'count by named entity'),
    ('Embeddings', 'WEAT-style associations baked in',       'WEAT on entity pairs'),
    ('Retrieval',  'biased ranking of relevant chunks',     'name-swap retrieval test'),
    ('Re-ranker',  'LLM-as-judge with model bias',           'paired counterfactual eval'),
    ('Generation', 'model defaults override context',        'final answer comparison'),
]:
    print(f'  {stage:14s}: {source_of_bias:38s} | audit: {audit_method}')
print()
print('Worst stage dominates -- audit all five.')


## Ex 7: Mitigation Layers + Trade-Offs


In [ ]:
print('Mitigation techniques (layer them):')
for tech, effect, trade_off in [
    ('Counterfactual data aug',     'reduces stereotype baked-in', 'expensive labeling'),
    ('RLHF for stereotyping refusal','model declines harmful prompts','over-refusal risk'),
    ('Output classifier filter',     'blocks harmful outputs',     'reduces coverage, false +ve'),
    ('System prompt rules',          'cheap, immediate',           'jailbreakable'),
    ('Re-ranker fairness constraint','demographic parity in chunks','reduces top-k quality'),
    ('Eval-on-prod 1% sampler',     'continuous detection',       'paid judge calls'),
]:
    print(f'  {tech:30s}: {effect:32s} | cost: {trade_off}')


---
## Recap
Bias = signal; unfairness = harm. Test stereotype + sentiment + representational + quality + refusal.
Use chi-squared + bootstrap CI; pre-register thresholds.
Indian context: caste, religion, region, language, skin tone, intersectional.
Audit each RAG stage; layer mitigations. Monitor 1% in prod with drift alerts.
